# Notebook: 00 Preprocess Images
### Purpose: Convert TIFF files to PNG for stable training and validate image integrity


In [1]:
import os
import sys


sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../src"))

from pathlib import Path

from src.data.image_loader import validate_image_directory
from src.utils.config import Config
from src.utils.helpers import init_notebook, p, t
from src.utils.image_converter import ImageConverter


config = Config.load()
init_notebook(config.train.seed)




=== init_notebook ===
Done


#### Step 1: Validate Current Images

In [2]:
t("Validating train images")
train_results = validate_image_directory(config.paths.train_images)


=== Validating train images ===
Validating images: 150 files
Valid images: 150
Invalid images: 0


In [3]:
if config.paths.eval_images and config.paths.eval_images.exists():
    t("Validating eval images")
    eval_results = validate_image_directory(config.paths.eval_images)


=== Validating eval images ===
Validating images: 150 files
Valid images: 150
Invalid images: 0


#### Step 2: Convert TIFF to PNG

Why convert?
- TIFF files (especially GeoTIFFs) can cause system crashes
- OpenCV has inconsistent TIFF support
- PNG is lossless, universally supported, and faster to load
- Your crashes are likely caused by TIFF decompression issues


In [4]:
t("Converting train images")

converter = ImageConverter(
        source_dir = config.paths.train_images,
        target_dir = config.paths.train_images
)

stats = converter.convert_batch(overwrite = False)


=== Converting train images ===
[Info]: Found 300 TIFF files


Converting TIFFs to PNG: 100%|██████████| 300/300 [00:00<00:00, 30567.76it/s]

=== Conversion Summary ===
[Info]: Total files: 300
[Info]: Converted: 0
[Info]: Skipped: 300
[Info]: Failed: 0


#### Step 3: Update Annotations

Automatically creates timestamped backups before modifying


In [5]:
if config.paths.annotations and Path(config.paths.annotations).exists():
    t("Updating annotations to reference PNG files")

    # Backups are created automatically with timestamps
    # e.g., annotations.json.backup_20241120_143022
    #       annotations.json.backup (latest)

    converter.update_annotations(
            annotations_path = config.paths.annotations,
            create_backup = True
    )


=== Updating annotations to reference PNG files ===
[Info]: Created backup: train_annotations.json.backup_20251129_100730
[Info]: Latest backup: train_annotations.json.backup
[Info]: Updated 0 file references in annotations


#### Step 4: Validate Converted Images


In [6]:
t("Validating converted images")
final_results = validate_image_directory(config.paths.train_images)


=== Validating converted images ===
Validating images: 150 files
Valid images: 150
Invalid images: 0


In [7]:
# Print summary
t("Conversion Summary")
p("Original valid", train_results["valid"])
p("Original invalid", train_results["invalid"])
p("Final valid", final_results["valid"])
p("Final invalid", final_results["invalid"])

if stats["failed"] > 0:
    t("Failed Conversions")
    for err in stats["errors"]:
        p(err["file"], err["error"])


=== Conversion Summary ===
Original valid: 150
Original invalid: 0
Final valid: 150
Final invalid: 0


#### Convert Eval Images


In [8]:
if config.paths.eval_images and config.paths.eval_images.exists():
    t("Converting eval images")

    eval_converter = ImageConverter(
            source_dir = config.paths.eval_images,
            target_dir = config.paths.eval_images
    )

    eval_stats = eval_converter.convert_batch(overwrite = False)


=== Converting eval images ===
[Info]: Found 300 TIFF files


Converting TIFFs to PNG: 100%|██████████| 300/300 [00:00<00:00, 34665.58it/s]

=== Conversion Summary ===
[Info]: Total files: 300
[Info]: Converted: 0
[Info]: Skipped: 300
[Info]: Failed: 0


#### Restore Annotations from Backup (if needed)

If something went wrong, restore the original annotations


In [9]:
# from src.utils.image_converter import ImageConverter
#
# success = ImageConverter.restore_annotations_from_backup(
#     annotations_path=config.paths.annotations
# )
#
# if success:
#     p("Annotations restored successfully")
# else:
#     p("Failed to restore annotations")
#
#     # List available backups
#     backup_dir = Path(config.paths.annotations).parent
#     backups = list(backup_dir.glob("*.backup*"))
#     p("Available backups", len(backups))
#     for backup in backups:
#         p("", backup.name)